# 🎧 Can you still hear it if you smear it in time?

A cloud of short tones, arriving at random. Hidden inside it, a small group of tones
keeps coming back **at the same pitches, over and over** — the *figure*. You find it
without trying, because those tones **start together**.

This notebook asks what happens when they stop starting together. We delay each tone of
the figure a little behind the one below it — **0 ms** (all at once, a chord), then 10,
20, … up to **50 ms** (one after another, a run up a keyboard) — and you listen for the
point where the figure stops being *one thing* and dissolves back into the cloud.

Nobody has measured this. Teki (2013) and O'Sullivan (2015) slid the figure around in
*frequency* and it survived; sliding it in *time* is the case where the textbook says it
should break.

---

**Before you start**

1. 🎧 **Headphones on, both ears.** Laptop speakers will not do — the figure lives in the
   quiet middle of the cloud.
2. ▶️ **Runtime → Run all** (or ⇧⏎ through the cells). Setup takes about 20 seconds.
3. 🔊 Turn it up to a comfortable *conversational* level and then leave the volume alone.

In [ ]:
# @title 🛠  Setup — fetch the code and warm up  (click ▶, ~20 s)
import os, subprocess, sys

REPO = "https://github.com/MeysamAmirsardari/Rate_RNN.git"
if not os.path.isdir("Rate_RNN"):
    # a sparse clone: the repo is 5 GB, the stimulus code is 60 kB
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "--filter=blob:none", "--sparse", REPO], check=True)
    subprocess.run(["git", "-C", "Rate_RNN", "sparse-checkout", "set", "--no-cone",
                    "/audios/*.py", "/audios/sfg/*.py", "/audios/sfg_task/*.py"],
                   check=True)
sys.path.insert(0, os.path.abspath("Rate_RNN"))

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import spectrogram as _spec
from IPython.display import Audio, HTML, display

from audios.sfg_task.config import Design
from audios.sfg_task.stimulus import make_pool, trial
from audios.sfg_task.plot import raster_ax, RED

D = Design()          # every parameter of the experiment lives in this one object
D.validate()
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

_pools = {}
def pool_for(d):
    """Channel grid for a design (cached — it never changes within a run)."""
    key = (d.f_lo, d.f_hi, d.grid_st, d.phon, d.min_sep_erb)
    if key not in _pools:
        _pools[key] = make_pool(d)
    return _pools[key]

def build(step_ms, seed=0, variant="rise", d=None):
    """One trial: the interval WITH the figure, and the matched one without."""
    d = d or D
    return trial(d, pool_for(d), step_ms=step_ms, seed=seed,
                 variant=variant, rove=False)

def player(clip, gain, fs=None):
    """One HTML5 audio player at a gain you choose, so levels stay comparable."""
    return Audio(clip * gain, rate=fs or D.fs, normalize=False)._repr_html_()

def gain_for(*clips):
    return 0.89 / max(float(np.abs(c).max()) for c in clips)

def players(clips, labels, fs=None):
    g = gain_for(*clips)
    return HTML("".join(
        f"<div style='margin:2px 0 12px'><b>{l}</b><br>{player(c, g, fs)}</div>"
        for c, l in zip(clips, labels)))

def score(schs, titles, d=None, height=2.9):
    """The score: every tone a dot, the figure's tones in red."""
    d = d or D
    fig, ax = plt.subplots(1, len(schs), figsize=(5.6 * len(schs), height),
                           sharey=True, squeeze=False, constrained_layout=True)
    for a, sch, t in zip(ax[0], schs, titles):
        raster_ax(d, pool_for(d), sch, a)
        a.set_title(t, fontsize=10)
        a.set_xlabel("time (s)")
    ax[0][0].set_ylabel(f"semitones re {d.f_lo:.0f} Hz")
    plt.show()

def spectro(y, title="", fs=None, top_db=55):
    """What the sound actually looks like, rather than what we asked for."""
    fs = fs or D.fs
    f, t, S = _spec(y, fs, nperseg=1024, noverlap=896)
    S = 10 * np.log10(S + 1e-20)
    fig, ax = plt.subplots(figsize=(11, 3), constrained_layout=True)
    ax.pcolormesh(t, f, S - S.max(), vmin=-top_db, vmax=0,
                  cmap="magma", shading="auto")
    ax.set_yscale("log"); ax.set_ylim(D.f_lo * .9, D.f_hi * 1.1)
    ax.set_ylabel("frequency (Hz)"); ax.set_xlabel("time (s)"); ax.set_title(title)
    plt.show()

print("✅ ready\n")
print(D.summary())

---
## 1 · The cloud

First, the thing the figure has to hide in.

Eight tones are sounding at any instant, each 50 ms long, drawn from 117 pitches spread
over five octaves. Two rules keep it pleasant rather than gritty: **no two tones ever
sound at once inside the same critical band** (otherwise they beat against each other and
you hear a warble that has nothing to do with the experiment), and every tone is weighted
to be **equally loud** rather than equally powerful.

The plot below is the *score* — one dot per tone, time across, pitch up. The red dots are
tones that in this interval land somewhere new every time, so there is nothing to latch
onto. Listen: it should sound like rain on a window, with no shape in it.

In [ ]:
_, cloud = build(step_ms=20, seed=3)          # the figure-absent interval
display(players([cloud["y"]], ["🌧  the cloud, nothing hiding in it"]))
score([cloud], ["no figure — every group lands somewhere new"])
spectro(cloud["y"], "the same six seconds, as a spectrogram")

---
## 2 · The figure

Now the thing you are hunting for. Seven tones, always the **same seven pitches**, all
starting at the same moment — a chord — repeating eleven times across the six seconds at
an irregular rhythm.

Hear it **on its own** first, then hear the identical seven tones buried in the cloud.
Once you know what to listen for it is hard to un-hear.

In [ ]:
figure, nofigure = build(step_ms=0, seed=11)
g = gain_for(figure["y"], nofigure["y"], figure["y_fig"])
display(HTML(
    f"<div style='margin:2px 0 12px'><b>🎯 the figure alone</b> — "
    f"exactly the tones you are looking for<br>{player(figure['y_fig'], g)}</div>"
    f"<div style='margin:2px 0 12px'><b>🔍 the same figure, in the cloud</b>"
    f"<br>{player(figure['y'], g)}</div>"
    f"<div style='margin:2px 0 12px'><b>🌧 and the matched interval with no figure</b>"
    f"<br>{player(nofigure['y'], g)}</div>"))
score([figure, nofigure], ["figure present — the red rows repeat",
                           "figure absent — same rhythm, new pitches each time"])

---
## 3 · Smear it in time  ⏳

Here is the whole experiment in one cell.

Each row delays every tone of the figure a bit further behind the one below it. The red
dots tip over from a vertical chord to a diagonal staircase. **Nothing else changes** —
the same seven pitches, the same eleven repetitions, the same number of tones in the
cloud, the same loudness.

Start at the top and work down. Somewhere in this list the figure stops being an event
you notice and becomes something you have to hunt for. **Where does it go for you?**

In [ ]:
rows, cells_html = [], []
for step in D.steps_ms:
    fig_iv, _ = build(step_ms=step, seed=21)
    g = gain_for(fig_iv["y"], fig_iv["y_fig"])
    label = "all at once (a chord)" if step == 0 else \
            f"{D.extent_ms(step):.0f} ms from first tone to last"
    cells_html.append(
        f"<tr><td style='padding:6px 14px;white-space:nowrap'>"
        f"<b style='font-size:15px'>{step:g} ms</b><br>"
        f"<span style='color:#888;font-size:11px'>{label}</span></td>"
        f"<td style='padding:6px'>{player(fig_iv['y'], g)}</td>"
        f"<td style='padding:6px'>{player(fig_iv['y_fig'], g)}</td></tr>")
    rows.append(fig_iv)

display(HTML(
    "<table style='border-collapse:collapse'><tr>"
    "<th style='text-align:left;padding:4px 14px'>delay per tone</th>"
    "<th style='text-align:left;padding:4px'>in the cloud</th>"
    "<th style='text-align:left;padding:4px'>the figure alone</th></tr>"
    + "".join(cells_html) + "</table>"))

fig, ax = plt.subplots(1, len(rows), figsize=(2.4 * len(rows), 3.0),
                       sharey=True, constrained_layout=True)
for a, sch, step in zip(ax, rows, D.steps_ms):
    raster_ax(D, pool_for(D), sch, a)
    a.set_title(f"{step:g} ms", fontsize=10); a.set_xlabel("s")
ax[0].set_ylabel(f"semitones re {D.f_lo:.0f} Hz")
plt.show()

---
## 4 · Your turn — which one had the figure? 🎲

This is the actual experiment. Two intervals, exactly one of them contains a figure, and
they are matched in every way we know how to match them: same rhythm, same number of
tones, same loudness to a hundredth of a decibel.

Listen to **A**, then **B**, make your call, *then* open the answer.
**Re-run this cell for a new pair** — the step size is drawn at random each time.

In [ ]:
import random

step = random.choice(D.steps_ms)
present, absent = build(step_ms=step, seed=random.randrange(10 ** 6))
first = random.choice([0, 1])
ivs = [present, absent] if first == 0 else [absent, present]
g = gain_for(ivs[0]["y"], ivs[1]["y"], present["y_fig"])

answer = "AB"[first]
shape = "a chord, all at once" if step == 0 else (
    "a staircase %.0f ms long" % D.extent_ms(step))

display(HTML(
    "<div style='margin:2px 0 12px'><b>Interval A</b><br>"
    + player(ivs[0]["y"], g) + "</div>"
    "<div style='margin:2px 0 12px'><b>Interval B</b><br>"
    + player(ivs[1]["y"], g) + "</div>"
    "<details style='margin-top:10px;padding:12px 16px;background:#f4f4f6;"
    "border-radius:8px;max-width:560px'>"
    "<summary style='cursor:pointer;font-weight:600'>&#128064; &nbsp;Reveal</summary>"
    "<p style='margin:10px 0 4px'>The figure was in "
    "<b style='font-size:16px'>interval " + answer + "</b>.</p>"
    "<p style='margin:4px 0;color:#555'>Its tones were delayed <b>"
    + ("%g" % step) + " ms</b> apart &mdash; " + shape + ".<br>"
    "Here it is alone; then play interval " + answer + " again.</p>"
    + player(present["y_fig"], g)
    + "<p style='margin:10px 0 0;color:#888;font-size:12px'>"
    "Re-run the cell for another pair.</p></details>"))
score(ivs, ["Interval A", "Interval B"])

---
## 5 · Playground 🎛

Everything is one object, `Design`, so you can turn any of it and hear the result.
Move the sliders and press **Run Interact**.

| knob | what it does |
|---|---|
| **delay per tone** | 0 ms is a chord, 50 ms is a staircase — the experiment's one variable |
| **tones in the figure** | fewer tones is a harder task; this is the classic difficulty knob |
| **repetitions** | how many times the figure comes back in six seconds |
| **cloud density** | tones sounding at any instant. Thin it out and the figure jumps out |
| **tone length** | short tones are clicky, long tones smear together |
| **direction** | a staircase going up, or coming down |

Some combinations are impossible — a very long figure repeated very often does not fit in
the interval — and it will say so rather than quietly giving you something else.

In [ ]:
import ipywidgets as W

@W.interact_manual(
    step_ms=W.SelectionSlider(options=[0, 5, 10, 20, 30, 40, 50], value=20,
                              description="delay/tone"),
    coherence=W.IntSlider(value=7, min=2, max=12, description="figure tones"),
    events=W.IntSlider(value=11, min=3, max=14, description="repetitions"),
    bg_sounding=W.IntSlider(value=8, min=3, max=14, description="cloud density"),
    tone_ms=W.SelectionSlider(options=[25, 50, 75], value=50, description="tone (ms)"),
    order=W.Dropdown(options=["rise", "fall"], value="rise", description="direction"),
    seed=W.IntSlider(value=0, min=0, max=99, description="seed"))
def playground(step_ms, coherence, events, bg_sounding, tone_ms, order, seed):
    d = D.replace(
        coherence=coherence, events=events, bg_sounding=bg_sounding,
        tone_ms=float(tone_ms), order=order,
        steps_ms=tuple(sorted({0.0, float(step_ms), 50.0})))
    try:
        d.validate()
        present, absent = build(step_ms=float(step_ms), seed=seed, d=d)
    except ValueError as e:
        return print(f"🙅 that combination will not build — {e}")
    g = gain_for(present["y"], absent["y"], present["y_fig"])
    display(HTML(
        f"<b>{coherence} tones, {step_ms:g} ms apart "
        f"({d.extent_ms(step_ms):.0f} ms per repetition), {events}x in "
        f"{d.interval_s:g} s, {bg_sounding} tones of cloud</b>"
        f"<div style='margin:8px 0'>🎯 figure alone{player(present['y_fig'], g, d.fs)}</div>"
        f"<div style='margin:8px 0'>🔍 in the cloud{player(present['y'], g, d.fs)}</div>"
        f"<div style='margin:8px 0'>🌧 matched, no figure{player(absent['y'], g, d.fs)}</div>"))
    score([present, absent], ["figure", "no figure"], d=d)

---
## 6 · The controls 🧪

If detection falls off as the figure is smeared, a referee will ask three questions. Each
has a stimulus that answers it, and you can hear all three here at a 20 ms delay.

- **shuffled** — the same seven delays, but not in order, so it is no longer a rising
  sweep. *Is the effect about asynchrony, or about the tune it plays?*
- **unfrozen** — the same seven pitches arriving in the same window, but the delays are
  redrawn every repetition, so there is no fixed pattern to learn. *Does the pattern
  matter, or only that the pitches come back?*
- **scattered** — the same seven pitches at the same rate, never grouped into repetitions
  at all. Its long-term spectrum is **identical** to the figure's. *This is the one that
  separates temporal coherence from the figure just being spectrally prominent.*

In [ ]:
CONTROLS = [("rise",    "▶️ the figure itself (20 ms staircase)"),
            ("perm",    "🔀 shuffled — same delays, wrong order"),
            ("redraw",  "🎲 unfrozen — delays redrawn every repetition"),
            ("scatter", "💨 scattered — same pitches, never grouped")]

built = [build(step_ms=20, seed=5, variant=v)[0] for v, _ in CONTROLS]
g = gain_for(*[b["y"] for b in built])
display(HTML("".join(
    f"<div style='margin:2px 0 12px'><b>{name}</b><br>{player(b['y'], g)}</div>"
    for (v, name), b in zip(CONTROLS, built))))
score(built, [n.split(" ", 1)[1] for _, n in CONTROLS], height=2.6)

---
## 7 · What is held constant 🔬

The whole design rests on one claim: **the two intervals differ in coherence and in
nothing else.** That is not something to assert — it is measured, on freshly built
stimuli, every time. This is the table that goes in the supplement.

Read the rows as *figure / no figure*. The three lines at the bottom are the differences
that would be cues if they were not zero; the last line is what the same measurement
returns when there is nothing to find, which is how you know the line above it is noise.

(About 30 seconds.)

In [ ]:
from audios.sfg_task.verify import verify, table
from audios.sfg_task.plot import envelopes

res = [verify(D, s, n=6) for s in D.steps_ms]
print(table(res))

fig, ax = plt.subplots(1, len(res), figsize=(2.1 * len(res), 2.6),
                       sharey=True, constrained_layout=True)
for a, r in zip(ax, res):
    t = np.linspace(-150, 500, r["epoch"][0].size)
    a.plot(t, r["epoch"][0], color=RED, lw=1.1)
    a.plot(t, r["epoch"][1], color="k", lw=1.1, ls="--")
    a.axvline(0, color="0.75", lw=.8); a.set_title(f"{r['step_ms']:g} ms", fontsize=9)
    a.set_xlabel("ms re repetition")
    for s in ("top", "right"): a.spines[s].set_visible(False)
ax[0].set_ylabel("loudness (dB)")
plt.suptitle("loudness around each repetition — red = figure, black = no figure, "
             "on a ±0.1 dB axis", fontsize=9)
plt.show()

---
## Where this goes next

What you have been doing by ear is the experiment. Run properly it is **140 trials over
about 35 minutes**, two intervals a trial, seven delays, with a practice block, breaks,
and a level calibration — and it comes out as a psychometric curve whose one number is
*the delay at which the figure stops binding*.

```bash
git clone https://github.com/MeysamAmirsardari/Rate_RNN.git
cd Rate_RNN
python -m audios.sfg_task check          # the measurements from section 7
python -m audios.sfg_task calibrate      # set 65 dB SPL once
python -m audios.sfg_task run  S01       # the experiment; resumes if interrupted
python -m audios.sfg_task analyse S01    # d', the fit, the threshold
```

The full rationale, the control battery, and an honest list of what is *not* controlled
are in [`audios/sfg_task/README.md`](https://github.com/MeysamAmirsardari/Rate_RNN/blob/main/audios/sfg_task/README.md).

*Made with the same code that generates the real stimuli — nothing here is a mock-up.*